# VMT Reduction Likelihood Analysis

The purpose of this script is to define a set of rules that classify which trips in the TBI could, with high likelihood, switch to another mode. These trips will be a subset of the feasible trips and will satisfy more stringent conditions than the feasibility analysis, as even if a switch is feasible, that does not mean a person is likely to do it. The result of this analysis will be a set of binary flags attached to each TBI record where the mode is car:

    likely_walk_shift    - it is feasible for the trip to switch to walk
    likely_bike_shift    - it is feasible for the trip to switch to bike
    likely_transit_shift - it is feasible for the trip to switch to transit
    likely_shift         - it is feasible for the trip to switch to any non-car mode

In all cases, the flag starts as 1 (feasible) and we set it to infeasible if it violates one or more of our rules.  Even though our interst is in understanding which car trips could switch to another mode, we code this flag for all modes.  This way, we can see how existing trips on that mode violate the feasibility rules.  In general, we aim to follow the "95% rule" such that 95% of trips that currently use that mode meet the feasibility condition.  Therefore, about 5% of trips will violate our own rules.  This is ok, because these trips are the most dedicated users of that mode, and new users will likely be less dedicated.  

The list below shows several of the factors we wish to test.  

### Definition of what is a likely mode shift
- duration differences (15 minute threshold to start with w.r.t to the car trip)
- weather for walking (no walking when there is rain)
- weather for biking (no biking when there is rain)
- number of transfers (0 is likely)
- temperature > 12 degrees (walking and biking)
- light outside (walking and biking)
- shopping purpose/escort purpose should only be car
- disability -- only transit is likely
- any lts 3 or 4 distance -- biking unlikely
- timing without any overlaps in trips

In [41]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

In [42]:
import math
import keyring
import itertools
from ast import literal_eval

In [43]:
# This is used to avoid hard-coding directories. 
# To set the directory, use the command prompt or a notebook you don't check in.  run:
# import keyring
# keyring.set_password("msp", "vmt_reduction_dir", <directory>)

# get base path for data
data_dir = keyring.get_password("msp", "vmt_reduction_dir")

In [44]:
raw = pd.read_csv("feasibility_out.csv")

C:\Users\yianz\AppData\Local\Temp\ipykernel_1832\2786893731.py:1: DtypeWarning: Columns (34,35,36,53,65,66,67,68,69,70,71) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv("feasibility_out.csv")


In [45]:
# only consider trips that are a feasible shift; feasibility is a necessity for a trip being likely
df = raw.loc[raw["feasible_shift"], :].copy()

In [46]:
# reading in weather data
# https://www.ncei.noaa.gov/pub/data/ghcn/daily/
weather = pd.read_csv("extra_data/USW00014922.csv")
weather = weather[["Date", "Measurement", "Value"]]
weather = weather.pivot(index="Date", columns="Measurement", values="Value")
weather

C:\Users\yianz\AppData\Local\Temp\ipykernel_1832\2067923688.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  weather = pd.read_csv("extra_data/USW00014922.csv")


Measurement,ACMH,ACSH,ADPT,ASLP,ASTP,AWBT,AWND,FMTM,FRGT,PGTM,...,WT16,WT17,WT18,WT19,WT21,WT22,WV01,WV03,WV07,WV20
Date,,,,,,,,,,,,,,,,,,,,,
19380409,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19380410,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19380411,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19380412,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19380413,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20230311,NaN,NaN,NaN,NaN,NaN,NaN,57.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20230312,NaN,NaN,NaN,NaN,NaN,NaN,54.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20230313,NaN,NaN,NaN,NaN,NaN,NaN,39.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
weather["year"] = weather.index.astype(str).str[0:4]
weather["month"] = weather.index.astype(str).str[4:6]
weather["day"] = weather.index.astype(str).str[6:8]
weather["date"] = weather["year"] + "-" + weather["month"] + "-" + weather["day"]
weather = weather.set_index("date")
weather

Measurement,ACMH,ACSH,ADPT,ASLP,ASTP,AWBT,AWND,FMTM,FRGT,PGTM,...,WT19,WT21,WT22,WV01,WV03,WV07,WV20,year,month,day
date,,,,,,,,,,,,,,,,,,,,,
1938-04-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1938,04,09
1938-04-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1938,04,10
1938-04-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1938,04,11
1938-04-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1938,04,12
1938-04-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1938,04,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-03-11,NaN,NaN,NaN,NaN,NaN,NaN,57.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,03,11
2023-03-12,NaN,NaN,NaN,NaN,NaN,NaN,54.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,03,12
2023-03-13,NaN,NaN,NaN,NaN,NaN,NaN,39.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,03,13


In [48]:
#| echo: true
# everything is feasible initially
df['likely_walk_shift'] = True
df['likely_bike_shift'] = True
df['likely_transit_shift'] = True
df['likely_shift'] = True

In [49]:
#| echo: true
# omit schools bus and taxi/ridehail/carshare -- very different from other modes
valid_modes = [("Car","distance"), 
               ("Bike/Scooter","distance"), 
               ("Walk","distance"), 
               ("Transit","distance")]

In [50]:
def plot_density(column: pd.Series, percentile=0.95, discrete=False, bins=100, size=(12, 6)):
    fig, ax = plt.subplots(figsize=size)
    sns.histplot(column, ax=ax, discrete=discrete, bins=bins, kde=True, stat="density")
    val = column.quantile(q=percentile)
    plt.axvline(x=val, color="red")
    return fig, ax

In [51]:
def plot_mode_density(df: pd.DataFrame, modes=valid_modes, percentile=0.95, size=(12, 6), bins=300, function=lambda x: x):
    palette = itertools.cycle(sns.color_palette()) # cycle through colors to make sure each mode gets a unique one
    fig, ax = plt.subplots(figsize=size)
    for m in modes: # cycle through all modes
        mode = m[0]
        column = m[1]
        label = mode + ' ' + column
        
        c = next(palette) # get color to use
        group = df[df["mode"] == mode] # filter out the current mode
        sns.histplot(function(group[column]), ax=ax, stat="density", kde=True, label=label, color=c, bins=bins) # plot hist plot with kde overlayed in the color
        val = function(group[column]).quantile(q=percentile) # calculate the value of the given percentile (default 0.95)
        plt.axvline(x=val, color=c) # plot line representing that value on the plot        
        plt.legend()
    return fig, ax

In [52]:
def show_summaries(df: pd.DataFrame, modes=valid_modes, percentile=[0.95]): # show normal summaries for each mode side by side
    res = []
    labels = []
    if type(percentile) != type([]):
        percentile = [percentile]
    p = [0.25, 0.5, 0.75]
    p += percentile
    p = list(set(p))
    for m in modes:        
        mode = m[0]
        column = m[1]
        labels.append(mode + ' ' + column)
        
        group = df[df["mode"] == mode]
        res.append(group[column].describe(percentiles=p))
    x = pd.concat(res, axis=1)
    x.columns = labels
    return x

## 1. Difference between durations of alternative mode and car mode > X minutes

Even if switching from a car mode to a non-car mode is considered feasible in terms of the factors discussed previously, a person may not be likely to choose to do so if the time difference is too great. Here we consider a 15 minute maximum.

### 1a. Bike duration difference (later, convert this cutoff to one based on the data)

In [53]:
df["duration_minus_bike"] = df["duration"] - df["bike_duration_seconds"] / 60
df[df["mode"] == "Car"]["duration_minus_bike"].describe(percentiles=[0.25, 0.5, 0.75, 0.8, 0.85, 0.9, 0.95])

count    202874.000000
mean        -67.463724
std          62.403613
min       -1499.788333
25%         -91.931250
50%         -51.450833
75%         -27.348333
80%         -22.668333
85%         -17.723333
90%         -11.855000
95%          -3.455000
max         849.526667
Name: duration_minus_bike, dtype: float64

In [54]:
#| output: true

# require all bike trips to occur via 250 meters or less on lts 3/4 roads
df["likely_biking_car_duration_diff"] = True

percent_before = len(df[(df['mode']=='Car') & (df['likely_bike_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to biking.")

df.loc[df["duration_minus_bike"] < -15, 'likely_bike_shift'] = False
df.loc[df["duration_minus_bike"] < -15, 'likely_biking_car_duration_diff'] = False

percent_after = len(df[(df['mode']=='Car') & (df['likely_bike_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to biking.")


Before constraint  100.0  percent of car trips could shift to biking.
After constraint  12.520579275806659  percent of car trips could shift to biking.


### 1b. Walk duration difference (later, convert this cutoff to one based on the data)

In [55]:
df["duration_minus_walk"] = df["duration"] - df["walk_duration_seconds"] / 60
df[df["mode"] == "Car"]["duration_minus_walk"].describe(percentiles=[0.25, 0.5, 0.75, 0.8, 0.85, 0.9, 0.95])

count    202874.000000
mean        -51.754845
std          63.913270
min       -1106.426667
25%         -73.313333
50%         -31.442500
75%         -11.698333
80%          -8.548333
85%          -5.276667
90%          -1.883333
95%           2.895000
max         850.415000
Name: duration_minus_walk, dtype: float64

In [56]:
#| output: true

df["likely_walking_car_duration_diff"] = True

percent_before = len(df[(df['mode']=='Car') & (df['likely_walk_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to biking.")

df.loc[df["duration_minus_walk"] < -15, 'likely_walk_shift'] = False
df.loc[df["duration_minus_walk"] < -15, 'likely_walking_car_duration_diff'] = False

percent_after = len(df[(df['mode']=='Car') & (df['likely_walk_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to biking.")


Before constraint  100.0  percent of car trips could shift to biking.
After constraint  30.223192720604906  percent of car trips could shift to biking.


### 1c. Transit duration difference (later, convert this cutoff to one based on the data)

In [57]:
df["duration_minus_transit"] = df["duration"] - df["transit_duration"] / 60
df[df["mode"] == "Car"]["duration_minus_transit"].describe(percentiles=[0.25, 0.5, 0.75, 0.8, 0.85, 0.9, 0.95])

count    202874.000000
mean         13.910485
std          18.905240
min         -36.271389
25%           6.116667
50%          10.200417
75%          16.214097
80%          18.375667
85%          20.819222
90%          24.948917
95%          32.985986
max         854.816667
Name: duration_minus_transit, dtype: float64

In [59]:
#| output: true

df["likely_transit_car_duration_diff"] = True

percent_before = len(df[(df['mode']=='Car') & (df['likely_transit_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to biking.")

df.loc[df["duration_minus_transit"] < -15, 'likely_transit_shift'] = False
df.loc[df["duration_minus_transit"] < -15, 'likely_transit_car_duration_diff'] = False

percent_after = len(df[(df['mode']=='Car') & (df['likely_transit_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to biking.")


Before constraint  100.0  percent of car trips could shift to biking.
After constraint  99.75600619103483  percent of car trips could shift to biking.


## 2. Timing

As discussed in the feasibility analysis, a trip was considered feasible timing-wise if the actor could work around all discretionary trips (possibly skipping them and doing them another time) to manage to reach all non-negotiable, non-discretionary trips. However, it is not necessarily the case that the actor would be likely to embark on a trip if they needed to abandon some of their daily activities like shopping or social visits. 

Therefore, we consider a trip likely if, even with the duration increases that come with switching to non-car modes, the daily acitivies can still be done without any overlaps/need to skip any of them to make to the next.

In [63]:
#| echo: true

fixed_purposes = ["Work", "Work-related", "Escort", "School", "School-related"]

In [64]:
def convert_to_minutes(str):
    hours, minutes, seconds = [int(x) for x in str.split(":")]
    return hours * 60 + minutes + seconds / 60

def evaluate_timing(df, alt_mode_times):
    return df.groupby(["wave", "person_id", "travel_date"]).apply(lambda x: evaluate_probable_timing(x, alt_mode_times))


def evaluate_probable_timing(chunk, alt_mode_times: str):
    # if there is only an inbound and outbound trip, don't need to worry about timing
    if len(chunk) == 2:
        return True
    leg_starts = chunk["depart_time"].apply(lambda x: convert_to_minutes(x)).values
    ref = leg_starts[0]
    # start times, starting by 0 and accounting for midnight wraparound with the mod function, for each leg of the complete tour
    leg_starts = [(x - ref) % 1440 for x in chunk["depart_time"].apply(lambda x: convert_to_minutes(x)).values]
    leg_durations = chunk["duration"].values
    # calculate end times relative to the start times using the duration category
    leg_ends = [(x + y) for (x, y) in zip(leg_starts, leg_durations)]
    # these are arrays indicating whether each leg of the complete tour is a fixed arrival/departure
    fixed_arrivals = chunk["d_purpose_category"].isin(fixed_purposes).values
    fixed_departures = chunk["o_purpose_category"].isin(fixed_purposes).values
    
    # sanity check; if the atlernative time for any of the legs can't be found, return False (means can't route it feasibly, usually for transit)
    # do this as preprocessing
    # if ~(chunk["trip_id"].isin(alt_mode_times.index).any()):
    #     return False
    # alternative durations for each of the legs
    alt_durations = chunk[alt_mode_times].values # make it a col in the dataframe to simplify things
    if -1 in alt_durations:
        return True
    
    # need to omit this for probable -- could have other, non-fixed trips in a tour
    # if fixed_arrivals.sum() <= 1:
    #     return True
    
    for i in range(1, len(chunk) - 1):# can always neglect trip 0 (just leave earlier) and the last trip (just arrive later)
        # if the current trip is fixed arrival
        if fixed_arrivals[i]:
            # for probable, consider ALL adjacent trips; not just adjacent trips that are fixed
            # if you would need to start before the last trip finished, not feasible
            if leg_ends[i] - alt_durations[i] < leg_ends[i - 1]: 
                return False
        # if the current trip is fixed departure
        if fixed_departures[i]:
            # if you would arrive after the next trip began, not feasible
            if leg_starts[i] + alt_durations[i] > leg_starts[i + 1]: 
                return False

    # feasible if nothing is weird
    return True

### 2a. Biking

In [70]:
likely_biking = evaluate_timing(df, "bike_duration")

In [71]:
likely_biking.sum() / len(likely_biking)

0.8316346756352383

In [73]:
likely_biking = likely_biking.reset_index().rename(columns={0: "likely_biking"})
df = df.merge(likely_biking, on=["wave", "person_id", "travel_date"], how="left")

In [74]:
df["likely_biking"].sum() / len(df["likely_biking"])

0.7251710958695093

In [76]:
#| output: true

df["likely_timing_biking"] = True

percent_before = len(df[(df['mode']=='Car') & (df['likely_bike_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to biking.")

df.loc[~df["likely_biking"], 'likely_bike_shift'] = False
df.loc[~df["likely_biking"], 'likely_timing_biking'] = False

percent_after = len(df[(df['mode']=='Car') & (df['likely_bike_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to biking.")


Before constraint  2.7874444236324023  percent of car trips could shift to biking.
After constraint  0.0  percent of car trips could shift to biking.


### 2b. Walking

In [77]:
likely_walking = evaluate_timing(df, "walk_duration")

In [78]:
likely_walking.sum() / len(likely_walking)

0.8568651773040576

In [79]:
likely_walking = likely_walking.reset_index().rename(columns={0: "likely_walking"})
df = df.merge(likely_walking, on=["wave", "person_id", "travel_date"], how="left")

In [80]:
df["likely_walking"].sum() / len(df["likely_walking"])

0.7619246098939745

In [81]:
#| output: true

df["likely_timing_walking"] = True

percent_before = len(df[(df['mode']=='Car') & (df['likely_walk_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to biking.")

df.loc[~df["likely_walking"], 'likely_walk_shift'] = False
df.loc[~df["likely_walking"], 'likely_timing_walking'] = False

percent_after = len(df[(df['mode']=='Car') & (df['likely_walk_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to biking.")


Before constraint  30.223192720604906  percent of car trips could shift to biking.
After constraint  23.907449944300403  percent of car trips could shift to biking.


### 2c. Transit

In [82]:
likely_transit = evaluate_timing(df, "transit_duration")

In [83]:
likely_transit.sum() / len(likely_transit)

0.9199350375324493

In [84]:
likely_transit = likely_transit.reset_index().rename(columns={0: "likely_transit"})
df = df.merge(likely_transit, on=["wave", "person_id", "travel_date"], how="left")

In [85]:
df["likely_transit"].sum() / len(df["likely_transit"])

0.8623641188051014

In [86]:
#| output: true

df["likely_timing_transit"] = True

percent_before = len(df[(df['mode']=='Car') & (df['likely_transit_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to biking.")

df.loc[~df["likely_transit"], 'likely_transit_shift'] = False
df.loc[~df["likely_transit"], 'likely_timing_transit'] = False

percent_after = len(df[(df['mode']=='Car') & (df['likely_transit_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to biking.")


Before constraint  99.75600619103483  percent of car trips could shift to biking.
After constraint  85.15926141348817  percent of car trips could shift to biking.
